# Day05 下午个人项目：电商用户多维分析

**姓名：** 钟俊坤
**专题方向：** A 

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "钟俊坤"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 钟俊坤
专题： A
输入数据： d:\大学\大二下\暑假实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： d:\大学\大二下\暑假实习\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 钟俊坤
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")


# TODO 3：展示验收结果
display(validation)

行数                 5630
列数                   22
CustomerID重复数         0
核心字段缺失数               0
Churn取值          [0, 1]
Name: 验收结果, dtype: object

In [5]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行代表一名独立用户，为用户级聚合数据

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：该字段是用户唯一标识符，不能作为普通连续数值参与后续分析

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [6]:
# TODO：计算公共基础指标

customer_count = df.shape[0]
churn_count = df["Churn"].sum()
churn_rate = churn_count/customer_count
average_order_count = df["OrderCount"].sum()/customer_count
median_order_count = df["OrderCount"].median()
average_couponused = df["CouponUsed"].sum()/customer_count
average_cashback_amount = df["CashbackAmount"].sum()/customer_count
average_hourspendonapp = df["HourSpendOnApp"].sum()/customer_count
average_satisfactionscore = df["SatisfactionScore"].sum()/customer_count
average_day_since_last_order = df["DaySinceLastOrder"].sum()/customer_count
overall_metrics = pd.DataFrame({
    "用户数":customer_count,
    "流失人数":churn_count,
    "流失率":churn_rate,
    "平均订单数":average_order_count,
    "订单中位数":median_order_count,
    "平均优惠券数":average_couponused,
    "平均返现":average_cashback_amount,
    "平均App时长":average_hourspendonapp,
    "平均满意度":average_satisfactionscore,
    "平均距上次下单天数":average_day_since_last_order,
    },
    index = [0]).T.reset_index()


# TODO：展示结果
display(overall_metrics)

,index,0
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [7]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = churn_rate

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：“当前样本共有5630名用户，总体流失率为0.17”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [8]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    平均订单数=("OrderCount", "mean"),
    平均优惠券数=("CouponUsed","mean"),
    平均返现金额=("CashbackAmount", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均满意度=("SatisfactionScore","mean"),
    流失率=("Churn", "mean")
).round(2)

# TODO 3：重置索引、按业务意义排序并展示
segment_analysis = segment_analysis.reset_index()
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,平均订单数,平均优惠券数,平均返现金额,平均App使用时长,平均满意度,流失率
0,0-6个月,1642,2.68,1.74,164.87,3.14,3.09,0.26
1,13-24个月,1467,3.70,2.02,204.92,2.94,3.09,0.06
2,24个月以上,429,3.55,1.94,222.34,2.87,3.05,0.00
3,7-12个月,1584,2.75,1.60,163.31,2.88,2.99,0.10
4,新用户,508,1.89,0.96,142.44,2.51,3.18,0.54


In [9]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期用户的流失和订单行为有何差异？

**数据现象：**

> TODO：
新用户有508名，流失率最高，达0.54；0-6个月用户规模最大，共1642名，流失率为0.26；24个月以上用户有429名，流失率最低，为0

**可能解释：**

> TODO：
1. 新用户流失率高可能与产品体验未达预期相关，需进一步验证首单转化流程是否存在问题。
2. 长期用户留存率高可能与用户粘性积累有关，相关因素可能包括服务体验、优惠激励等。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [10]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "PreferedOrderCat"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2],observed=True).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均优惠券数=("CouponUsed", "mean"),
    平均返现金额=("CashbackAmount", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均满意度=("SatisfactionScore", "mean")
).round(2)


# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO 4：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values(by="流失率", ascending=False).reset_index()
display(cross_analysis)

,TenureGroup,PreferedOrderCat,用户数,流失人数,流失率,平均订单数,平均优惠券数,平均返现金额,平均App使用时长,平均满意度,样本提示
0,0-6个月,Others,5,4,0.80,12.80,4.60,307.97,2.60,3.00,小样本
1,新用户,Fashion,40,32,0.80,3.00,1.55,194.61,2.35,3.45,可观察
2,新用户,Mobile Phone,311,168,0.54,1.69,0.89,129.02,2.60,3.12,可观察
3,0-6个月,Grocery,25,12,0.48,4.16,2.68,259.40,2.88,3.24,小样本
4,新用户,Laptop & Accessory,155,72,0.46,1.95,0.95,154.18,2.35,3.21,可观察
5,7-12个月,Others,10,4,0.40,3.60,2.10,296.10,2.60,2.70,小样本
6,0-6个月,Mobile Phone,816,273,0.33,2.35,1.57,147.29,3.27,3.15,可观察
7,0-6个月,Fashion,163,36,0.22,3.84,2.33,211.78,2.98,3.14,可观察
8,7-12个月,Grocery,21,4,0.19,7.67,2.95,259.32,2.95,3.43,小样本
9,0-6个月,Laptop & Accessory,633,100,0.16,2.68,1.74,170.60,3.01,3.00,可观察


In [11]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：新用户 + Fashion（时尚品类）组合

**该组合的用户数、流失率和比较对象：**

> TODO：样本量：40人（流失人数32人），流失率：80%，与新用户+Mobile Phone（流失率0.54）相比，流失率明显偏高

**是否存在小样本风险：**

> TODO：样本量虽标记为"可观察"，但40人仍属于较小样本，结论需在更大样本中验证。

**为什么不能直接写成因果结论：**

> TODO：相关不等于因果：数据仅显示"新用户+Fashion"与"高流失率"存在相关性，不能证明存在直接因果关系。

## 任务5：输出统计报表（必做）

In [12]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [13]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 8)
通过：cross_analysis.csv，形状为(25, 11)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户中，流失率指标为54%，与24个月以上用户（0%）相比高出54个百分点。对应证据表：segment_analysis（单维专题分析表）。

### 结论2

> TODO：在新用户中偏好购买Fashion品类的用户群体，流失率高达80%，显著高于新用户整体54%的流失率，是高流失风险细分人群。对应证据表：cross_analysis（双维度交叉分析表）。

### 结论3

> TODO：用户生命周期越长，平均订单数和平均返现金额越高：13-24个月用户平均订单数3.70单、平均返现204.92，分别是新用户（1.89单、142.44）的1.96倍和1.44倍。对应证据表：segment_analysis（单维专题分析表）。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：数据为用户级聚合数据，无法分析订单级的行为细节（如购买频次、复购周期等）；部分细分组合样本量较小（如0-6个月+Others仅5人），统计结论的可靠性受影响。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：
建议： 针对新用户中偏好Fashion品类的高流失风险群体，实施定向挽留策略，如首单后72小时内推送专属优惠券；
验证方式：将符合条件的用户随机分为两组，实验组实施挽留策略，对照组保持常规运营；
**所需数据**：需要额外收集用户分组后的行为数据（如是否领取优惠券、是否再次下单等）以及流失状态的后续追踪数据

## 拓展任务（选做）

In [14]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）
# 按订单数将用户分为5个活跃度等级
# 查看订单数的分布情况
df = pd.read_csv(DATA_PATH , encoding="utf-8-sig")
print("订单数分布：")
print(df["OrderCount"].value_counts().sort_index())

df["ActivityLevel"] = pd.cut(
    df["OrderCount"], 
    bins=[0, 1, 2, 3, 5, float('inf')],
    labels=["极低活跃", "低活跃", "中活跃", "高活跃", "极高活跃"]
)

# 分析各活跃度等级
activity_analysis = df.groupby("ActivityLevel", observed=True).agg(
    用户数=("CustomerID", "count"),
    流失率=("Churn", "mean"),
    平均优惠券数=("CouponUsed", "mean"),
    平均返现金额=("CashbackAmount", "mean")
).round(2).reset_index()

display(activity_analysis)

订单数分布：
OrderCount
1.00     1751
2.00     2283
3.00      371
4.00      204
5.00      181
6.00      137
7.00      206
8.00      172
9.00       62
10.00      36
11.00      51
12.00      54
13.00      30
14.00      36
15.00      33
16.00      23
Name: count, dtype: int64


,ActivityLevel,用户数,流失率,平均优惠券数,平均返现金额
0,极低活跃,1751,0.18,0.52,154.19
1,低活跃,2283,0.17,1.55,183.49
2,中活跃,371,0.18,1.91,174.60
3,高活跃,385,0.11,2.40,194.11
4,极高活跃,840,0.16,4.26,201.62


## 最终检查：GitHub提交前验收

In [15]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

1.用户流失率随使用时间的增长而降低  
2.第五个检查点，可以帮我判断输出文件是否齐全  
3.结论3（用户周期越长，平均订单数和返现越高），这里有相关关系，但不能说明因果关系，不同用户的消费能力不同  
4.如果增加一个字段，我最希望增加用户消费能力的字段，以此判断不同消费群体的消费特征和潜力  
5.将用户生命周期与流失率的交叉分析表转化为图表，以可视化的方式展示用户生命周期与流失率的关系
